# 3교시 · 데이터 결합과 집계
### — 여러 표를 하나로 합쳐 요약하기

앞 시간에 필요한 것만 남겼습니다. 이제 그것을 **묶어서 요약**합니다.
엑셀의 **피벗테이블**과 **VLOOKUP** 에 해당하는 부분입니다.

**이 시간이 끝나면 할 수 있는 것**

1. 항목별로 묶어서 합계·평균을 낼 수 있다
2. 두 개의 표를 이어 붙일 수 있다
3. 비교 기준을 바꿔 가며 볼 수 있다
4. **같은 데이터로 정반대 결론이 나올 수 있다는 것을 안다**

### 오늘 쓰는 데이터 — 문구·가구 유통사 주문 내역

| | |
|---|---|
| **무엇** | 어느 문구·가구 유통사의 주문 내역 (Tableau 공식 샘플 데이터) |
| **기간** | 2023-01-03 ~ 2026-12-30 (4년치) |
| **크기** | 10,239행 × 21열 · 주문 5,111건 · 고객 804명 |
| **한 행은** | 주문이 아니라 **주문에 담긴 품목 하나**입니다 |
| **지역** | 미국(10,038) · 캐나다(201) |

**주요 열**

| 열 | 뜻 |
|---|---|
| `Order ID` · `Order Date` · `Ship Date` | 주문번호 · 주문일 · 배송일 |
| `Customer ID` · `Segment` | 고객 · 고객 유형(Consumer / Corporate / Home Office) |
| `Region` · `State/Province` · `City` | 지역(Central / East / South / West) · 주 · 도시 |
| `Category` · `Sub-Category` · `Product Name` | 대분류(3종) · 소분류(17종) · 제품명 |
| `Sales` · `Quantity` · `Discount` · `Profit` | 매출 · 수량 · 할인율 · 이익 |

> **결측치·이상치·중복값이 일부러 들어 있습니다.**
> 실무에서 받는 데이터가 그렇기 때문입니다. 손대지 않은 원본이 필요하면
> `superstore_orders_raw.csv` 를 쓰세요.

---

### 함께 쓰는 표 — 반품 목록

`superstore_returns.csv` · **296행 × 2열** (`Order ID` · `Returned`)

**반품된 주문번호만** 들어 있습니다. 반품되지 않은 주문은 이 표에 아예 없습니다.
그래서 주문 내역에 붙이면 반품 안 된 주문 쪽이 결측치가 됩니다.

---

### 함께 쓰는 표 — 지역 담당자

`superstore_people.csv` · **4행 × 2열** (`Regional Manager` · `Region`)

지역마다 담당자가 한 명씩 있는 작은 대응표입니다. 주문 내역에는 담당자 정보가
없으므로, 담당자별 실적을 보려면 이 표를 붙여야 합니다.

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders  = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
returns = pd.read_csv(BASE + 'superstore_returns.csv')
people  = pd.read_csv(BASE + 'superstore_people.csv')

# 2교시에서 배운 대로 중복부터 제거하고 시작합니다
orders = orders.drop_duplicates()

print(orders.shape)
orders.head(3)

---
# 3-1. groupby — 엑셀 피벗테이블

`groupby` 는 "이 열의 값이 같은 것끼리 묶어라"는 뜻입니다.
묶은 다음에는 **무엇을 계산할지** 알려 줘야 합니다.

```
orders.groupby('무엇으로묶을까')['무엇을계산할까'].어떻게()
```

In [ ]:
orders.groupby('Category')['Sales'].sum()

방금 만든 것이 엑셀 피벗테이블과 똑같습니다.

| 엑셀 | pandas |
|---|---|
| 행 영역에 `Category` | `groupby('Category')` |
| 값 영역에 `Sales` | `['Sales']` |
| 값 요약 = 합계 | `.sum()` |

## 계산 방법 바꾸기

In [ ]:
orders.groupby('Category')['Sales'].____.round(0)

In [ ]:
summary = orders.groupby('Category').agg(
    sales=('Sales', 'sum'),
    profit=('Profit', 'sum'),
    n_orders=('Order ID', 'nunique'),
).round(0)

summary

In [ ]:
summary['margin_pct'] = (summary['profit'] / summary['sales'] * 100).round(1)

summary

---
# 3-2. 두 가지 기준으로 묶기

`groupby` 에 열을 두 개 넣으면 2단으로 묶입니다. 다만 결과가 세로로 길게 나와
읽기 불편합니다. **행과 열로 펼치면** 훨씬 잘 보입니다.

## pivot_table — 표로 펼치기


In [ ]:
orders.pivot_table(
    index='Region',
    columns='Category',
    values='Sales',
    aggfunc='sum',
).round(0)

In [ ]:
# 3-3. 시계열 — 월 단위로 묶기
orders['year_month'] = orders['Order Date'].dt.to_period('M')

monthly_sales = orders.groupby('year_month')['Sales'].sum()

monthly_sales.tail(6).round(0)

In [ ]:
trend = pd.DataFrame({'sales': monthly_sales})

trend['mom_pct']    = trend['sales'].pct_change() * 100
trend['yoy_pct'] = trend['sales'].pct_change(____) * 100

trend.loc['2026-07':'2026-12'].round(1)

---
# 3-4. merge — 두 표를 이어 붙이기

**엑셀의 VLOOKUP** 에 해당합니다.

지금 `orders` 에는 반품 정보가 없습니다. 그건 `returns` 라는 별도의 표에 있습니다.

In [ ]:
merged = orders.merge(returns, on='Order ID', how='left')

print('붙이기 전:', len(orders))
print('붙인 후  :', len(merged))

merged[['Order ID', 'Sales', 'Returned']].head(3)

## `how` 가 무엇을 남길지 정합니다

| `how` | 무엇이 남나 | 엑셀로 치면 |
|---|---|---|
| `'left'` | **왼쪽 표는 전부**, 오른쪽은 맞는 것만 | VLOOKUP |
| `'inner'` | **양쪽 다 있는 것만** | 교집합 |
| `'outer'` | 양쪽 전부 | 합집합 |
| `'right'` | 오른쪽 표 전부 | 거꾸로 VLOOKUP |

`how='left'` 를 썼기 때문에 주문은 하나도 안 사라졌습니다.
반품 기록이 없는 주문은 `Returned` 가 **결측치**가 됩니다.

In [ ]:
print('Returned 결측치:', merged['Returned'].isna().sum())

# 결측치 = 반품 안 됨 이므로 'No' 로 채웁니다
merged['Returned'] = merged['Returned'].fillna('No')

merged['Returned'].value_counts()

In [ ]:
print('행 수 변화 :', len(orders), '->', len(merged))
print('매출 합 변화: {:,.0f} -> {:,.0f}'.format(orders['Sales'].sum(), merged['Sales'].sum()))

In [ ]:
merged = merged.merge(people, on='____', how='left')

merged[['Region', 'Regional Manager', 'Sales']].head(3)

---
# 3-5. 실습 — 오늘 배운 다섯 가지

아래 다섯 문제의 **빈칸(`____`)을 채우고 실행**하세요.
각 문제는 오늘 배운 기능 하나씩에 대응합니다.


In [ ]:
# 문제 1. groupby — 지역(Region)별 매출 합계를 구하세요.
orders.____('Region')['Sales'].sum().round(0)

In [ ]:
# 문제 2. agg — 고객 유형(Segment)별 매출의 합계·평균·건수를 한 번에 구하세요.
orders.groupby('Segment')['Sales'].____(['sum', 'mean', 'count']).round(0)

In [ ]:
# 문제 3. pivot_table — 행은 Segment, 열은 Category, 값은 이익(Profit) 합계인 표를 만드세요.
orders.____(
    index='Segment',
    columns='Category',
    values='Profit',
    aggfunc='sum',
).round(0)

In [ ]:
# 문제 4. merge — 주문 표에 담당자 표(people)를 Region 기준으로 붙이세요.
with_mgr = orders.____(people, on='Region', how='left')

print('붙이기 전:', len(orders))
print('붙인 후  :', len(with_mgr))
with_mgr[['Region', 'Regional Manager', 'Sales']].head(3)

In [ ]:
# 문제 5. 시계열 — 월별 매출의 전월 대비 변화율(%)을 구해 마지막 6개월을 보세요.
(monthly_sales.____() * 100).round(1).tail(6)

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 묶어서 합계 | `df.groupby('열')['값'].sum()` |
| 여러 계산 한 번에 | `df.groupby('열')['값'].agg(['sum','mean','count'])` |
| 열마다 다른 계산 | `df.groupby('열').agg(이름=('값','sum'))` |
| 두 기준으로 묶기 | `df.groupby(['열1','열2'])` |
| 표로 펼치기 | `df.pivot_table(index=, columns=, values=, aggfunc=)` |
| 표 잇기 | `df.merge(다른표, on='열쇠', how='left')` |
| 월 만들기 | `df['날짜'].dt.to_period('M')` |
| 전월 대비 | `.pct_change()` |
| 전년 동월 대비 | `.pct_change(12)` |

## 남길 것 세 가지

1. **묶는 열 · 계산할 열 · 계산 방법** — 집계는 이 세 자리를 채우는 일입니다
2. **`merge` 뒤에는 행 수를 확인한다** — 늘었으면 열쇠가 중복이고, 다 비었으면 열쇠가 안 맞는 것입니다
3. **비교 기준을 함께 적는다** — "매출이 늘었다"가 아니라 "전년 동월 대비 얼마 늘었다"로 적습니다

---

### 다음 시간

지금까지 계속 **합계와 평균**을 봤습니다.
다음 시간에는 이렇게 묻습니다 — **그 평균, 믿어도 됩니까?**
